In [1]:
import sys
import os
import pandas as pd
from sqlalchemy import text

In [2]:
sys.path.append(os.path.abspath('../src'))
from scrapers.google_news_scraper import GoogleNewsScraper
from utils.db import get_db_engine

In [3]:
engine = get_db_engine()
bot = GoogleNewsScraper()

In [4]:
print(" Membaca konfigurasi kata kunci...")
query = "SELECT keyword FROM config_keywords WHERE is_active = true"
df_config = pd.read_sql(query, engine)
keywords = df_config['keyword'].tolist()
print(f" Kata kunci aktif: {keywords}")

 Membaca konfigurasi kata kunci...
 Kata kunci aktif: ['Universitas Singaperbangsa Karawang', 'UNSIKA', 'Rektor UNSIKA']


In [5]:
all_news = []

for kw in keywords:
    print(f"\n Memulai scraping untuk: {kw}")
    df_hasil = bot.scrape(kw, max_items=20) # Ambil 5 berita dulu buat tes

    if not df_hasil.empty:
        # Tambahkan kolom keyword biar tau ini berita hasil pencarian apa
        df_hasil['keyword'] = kw
        all_news.append(df_hasil)

 Berhasil mengambil 20 berita.


In [6]:
if all_news:
    final_df = pd.concat(all_news, ignore_index=True)

    # --- [TAMBAHAN PENTING] ---
    # Buang berita duplikat berdasarkan URL
    print(f"Jumlah awal: {len(final_df)} berita")
    final_df.drop_duplicates(subset=['url'], keep='first', inplace=True)
    print(f"Jumlah bersih (setelah hapus duplikat): {len(final_df)} berita")
    # --------------------------

    print(f"\n💾 Menyimpan {len(final_df)} berita ke PostgreSQL...")

    try:
        final_df.to_sql('news_raw', engine, if_exists='append', index=False, method='multi')
        print(" SUKSES! Data tersimpan di tabel 'news_raw'.")
    except Exception as e:
        print(f" Masih ada error simpan: {e}")
        # Opsional: Print error biar tau kenapa
else:
    print(" Tidak ada berita yang ditemukan.")

Jumlah awal: 60 berita
Jumlah bersih (setelah hapus duplikat): 54 berita

💾 Menyimpan 54 berita ke PostgreSQL...
 SUKSES! Data tersimpan di tabel 'news_raw'.
